## Comparing Feed Forward Networks and CNNs with MNIST

In this notebook we compare the use of a feed forward network and a CNN to construct a classifier. 

In [ ]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
import numpy as np

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load and Prepare MNIST

In [ ]:
# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor()])

# Download and load the training data
train_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)

# Download and load the test data
test_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)

In [ ]:
train_x = train_set.data.reshape((60000, 28*28))
train_x = train_x.float() / 255
test_x = test_set.data.reshape((10000, 28 * 28))
test_x = test_x.float() / 255

In [ ]:
train_set

In [ ]:
# Number of classes
num_classes = train_set.targets.max() + 1  # Assuming classes are labeled from 0 to n-1

# Convert to one-hot encoding
train_y = F.one_hot(train_set.targets, num_classes=num_classes).float()
test_y = F.one_hot(test_set.targets, num_classes=num_classes).float()

In [ ]:
epochs = 200
batch_size = 2048

In [ ]:
history_dict = dict()

### Build Feed Forward Network

We first build a feed forward network with 200 neurons per layer

In [ ]:
# Define the neural network model as a subclass of nn.Module
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        # Define the first layer with 200 units, the input size is 28*28
        self.fc1 = nn.Linear(28 * 28, 200)
        # Define the second layer with 200 units
        self.fc2 = nn.Linear(200, 200)
        # Define the output layer with 10 units (for 10 classes)
        self.fc3 = nn.Linear(200, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Flatten the input tensor
        x = x.view(-1, 28 * 28)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        z = self.softmax(x)
        return x

In [ ]:
# Create an instance of the NeuralNetwork
model = NeuralNetwork().to(device)

Print out the summary of the network architecture

In [ ]:
print(model)

In [ ]:
# Define the optimizer (Adam in this case)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Define the loss function (categorical cross-entropy in this case)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train_x = train_x.to(device)
train_y = train_y.to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, 
                                           shuffle=True, drop_last=True)

test_x = test_x.to(device)
test_y = test_y.to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = torch.argmax(batch_y, dim=1)
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = torch.argmax(batch_y, dim=1)
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
history_dict['Feed Forward'] = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)

### Build a CNN

We build a CNN with $3 \times 3$ filters and MaxPooling

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 1 * 1, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = x.view(-1, 64 * 1 * 1) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)

In [ ]:
cnn_model = CNNModel().to(device)

In [ ]:
print(cnn_model)

In [ ]:
train_x = train_set.data.unsqueeze(dim=1)
train_x = train_x.float() / 255
test_x = test_set.data.unsqueeze(dim=1)
test_x = test_x.float() / 255

In [ ]:
train_x = train_x.to(device)
train_y = train_y.to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, 
                                           shuffle=True, drop_last=True)

test_x = test_x.to(device)
test_y = test_y.to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

In [ ]:
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

In [ ]:
history_dict['CNN'] = train_model(cnn_model, train_loader, test_loader,
                                        loss_fn, cnn_optimizer, epochs)

### Plot Validation Accuracy

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
epochs = range(1, epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
for ilr in history_dict:
    valacc_hist = history_dict[ilr]
    val_acc_values = valacc_hist['test_acc']
    ax.plot(epochs, val_acc_values, label = str(ilr))

ax.set_xlabel('Epochs')
ax.set_ylabel('Validation Accuracy')
ax.set_ylim(95, 100)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('TestMNISTCNNFF.png', dpi=300, bbox_inches='tight')